# Section 1: Email Security and Phishing Detection

This targeted experiment uses the SpamAssassin public corpus to compare TF-IDF with Logistic Regression against an LSTM. Spam is used as a practical proxy for malicious email; this limitation must be stated in the report.

Run the cells from top to bottom. All code required for this experiment is contained in this notebook.

## 1. Install dependencies

In [ ]:
%pip install -q joblib matplotlib numpy pandas scikit-learn torch

In [ ]:
import hashlib
import json
import os
import random
import re
import shutil
import sys
import tarfile
import urllib.request
from collections import Counter
from email import policy
from email.parser import BytesParser
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, average_precision_score,
    classification_report, confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score,
    roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from torch import nn
from torch.utils.data import DataLoader, Dataset

SEED = 42
TEST_SIZE = 0.15
VALIDATION_SIZE = 0.15
LSTM_EPOCHS = 6  # Change to 1 for a quick pipeline check.

COLAB = "google.colab" in sys.modules
ROOT = Path("/content/section_01_workspace") if COLAB else Path.cwd()
DATA_DIR = ROOT / "data/raw/spamassassin"
PROCESSED_DIR = ROOT / "data/processed/section_01"
MODELS_DIR = ROOT / "models/section_01"
RESULTS_DIR = ROOT / "reports/section_01"

for directory in (DATA_DIR, PROCESSED_DIR, MODELS_DIR, RESULTS_DIR / "metrics", RESULTS_DIR / "figures"):
    directory.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 2. Download and read the corpus

In [ ]:
BASE_URL = "https://spamassassin.apache.org/old/publiccorpus"
ARCHIVES = {
    "20030228_easy_ham.tar.bz2": "easy_ham",
    "20030228_spam.tar.bz2": "spam",
}

for archive_name, folder_name in ARCHIVES.items():
    archive_path = DATA_DIR / archive_name
    if not (DATA_DIR / folder_name).exists():
        if not archive_path.exists():
            urllib.request.urlretrieve(f"{BASE_URL}/{archive_name}", archive_path)
        with tarfile.open(archive_path, "r:bz2") as archive:
            archive.extractall(DATA_DIR)

In [ ]:
WHITESPACE = re.compile(r"\s+")
HTML_TAG = re.compile(r"<[^>]+>")

def decode_part(part):
    payload = part.get_payload(decode=True)
    if not isinstance(payload, bytes):
        return str(part.get_payload())
    charset = part.get_content_charset() or "utf-8"
    try:
        return payload.decode(charset, errors="replace")
    except LookupError:
        return payload.decode("utf-8", errors="replace")

def read_email(path):
    message = BytesParser(policy=policy.default).parsebytes(path.read_bytes())
    parts = []
    for part in message.walk():
        if part.get_content_maintype() == "text" and part.get_content_disposition() != "attachment":
            parts.append(decode_part(part))
    text = f"Subject: {message.get('subject', '')} Body: {' '.join(parts)}"
    return WHITESPACE.sub(" ", HTML_TAG.sub(" ", text)).strip()

records = []
for folder, label in (("easy_ham", 0), ("spam", 1)):
    for path in sorted((DATA_DIR / folder).rglob("*")):
        if path.is_file() and path.name.lower() != "cmds":
            records.append({"path": str(path), "text": read_email(path), "label": label})

emails = pd.DataFrame(records).drop_duplicates("text").reset_index(drop=True)
emails["sha256"] = emails["text"].map(lambda text: hashlib.sha256(text.encode()).hexdigest())
emails["class"] = emails["label"].map({0: "ham", 1: "spam"})

class_counts = emails["class"].value_counts().reindex(["ham", "spam"])
display(class_counts.rename("messages").to_frame())
class_counts.plot.bar(color=["#4C78A8", "#E45756"], rot=0, ylabel="Messages", title="Class distribution")
plt.show()

## 3. Create leakage-safe splits

In [ ]:
train_validation, test = train_test_split(
    emails, test_size=TEST_SIZE, stratify=emails["label"], random_state=SEED,
)
train, validation = train_test_split(
    train_validation,
    test_size=VALIDATION_SIZE / (1 - TEST_SIZE),
    stratify=train_validation["label"],
    random_state=SEED,
)
train, validation, test = [frame.reset_index(drop=True) for frame in (train, validation, test)]

manifest = pd.concat([
    frame[["path", "label", "sha256"]].assign(split=name)
    for name, frame in (("train", train), ("validation", validation), ("test", test))
], ignore_index=True)
manifest[["split", "path", "label", "sha256"]].to_csv(PROCESSED_DIR / "split_manifest.csv", index=False)
display(pd.crosstab(manifest["split"], manifest["label"]).rename(columns={0: "ham", 1: "spam"}))

## 4. Shared preprocessing and evaluation

In [ ]:
URL = re.compile(r"(?:https?://|www\.)\S+", re.I)
EMAIL = re.compile(r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b")
NON_WORD = re.compile(r"[^a-z0-9_]+")

def normalize(text):
    text = URL.sub(" urltoken ", text.lower())
    text = EMAIL.sub(" emailtoken ", text)
    return WHITESPACE.sub(" ", NON_WORD.sub(" ", text)).strip()

def evaluate(name, labels, probabilities, filename):
    predictions = (probabilities >= 0.5).astype(int)
    metrics = {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "f1": f1_score(labels, predictions, zero_division=0),
        "roc_auc": roc_auc_score(labels, probabilities),
        "average_precision": average_precision_score(labels, probabilities),
        "confusion_matrix": confusion_matrix(labels, predictions).tolist(),
        "classification_report": classification_report(
            labels, predictions, target_names=["ham", "spam"], output_dict=True, zero_division=0,
        ),
    }
    (RESULTS_DIR / "metrics" / f"{filename}.json").write_text(json.dumps(metrics, indent=2))

    figure, axes = plt.subplots(1, 3, figsize=(15, 4))
    ConfusionMatrixDisplay.from_predictions(
        labels, predictions, display_labels=["ham", "spam"], cmap="Blues", colorbar=False, ax=axes[0],
    )
    axes[0].set_title("Confusion matrix")
    false_positive_rate, true_positive_rate, _ = roc_curve(labels, probabilities)
    axes[1].plot(false_positive_rate, true_positive_rate)
    axes[1].plot([0, 1], [0, 1], "--", color="grey")
    axes[1].set(xlabel="False positive rate", ylabel="True positive rate", title=f"ROC AUC: {metrics['roc_auc']:.3f}")
    precision, recall, _ = precision_recall_curve(labels, probabilities)
    axes[2].plot(recall, precision)
    axes[2].set(xlabel="Recall", ylabel="Precision", title=f"Average precision: {metrics['average_precision']:.3f}")
    figure.suptitle(name)
    figure.tight_layout()
    figure.savefig(RESULTS_DIR / "figures" / f"{filename}-evaluation.png", dpi=180)
    plt.show()
    return metrics

## 5. TF-IDF and Logistic Regression

In [ ]:
classic_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        preprocessor=normalize, stop_words="english", ngram_range=(1, 2),
        max_features=20000, min_df=2, max_df=0.98, sublinear_tf=True,
    )),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)),
])
classic_model.fit(train["text"], train["label"])
classic_probabilities = classic_model.predict_proba(test["text"])[:, 1]
classic_metrics = evaluate(
    "TF-IDF Logistic Regression", test["label"].to_numpy(),
    classic_probabilities, "classic",
)
joblib.dump(classic_model, MODELS_DIR / "tfidf-logistic-regression.joblib")
display(pd.Series(classic_metrics).loc[["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]].to_frame("score"))

## 6. LSTM

In [ ]:
MAX_VOCABULARY = 20000
MAX_LENGTH = 300
BATCH_SIZE = 64

def tokenize(text):
    return normalize(text).split()

counts = Counter(token for text in train["text"] for token in tokenize(text))
vocabulary = {"<PAD>": 0, "<UNK>": 1}
vocabulary.update({token: index for index, (token, _) in enumerate(counts.most_common(MAX_VOCABULARY - 2), 2)})

def encode(text):
    tokens = [vocabulary.get(token, 1) for token in tokenize(text)[:MAX_LENGTH]] or [1]
    length = len(tokens)
    return tokens + [0] * (MAX_LENGTH - length), length

class EmailDataset(Dataset):
    def __init__(self, frame):
        encoded = [encode(text) for text in frame["text"]]
        self.tokens = torch.tensor([item[0] for item in encoded], dtype=torch.long)
        self.lengths = torch.tensor([item[1] for item in encoded], dtype=torch.long)
        self.labels = torch.tensor(frame["label"].to_numpy(), dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.tokens[index], self.lengths[index], self.labels[index]

class EmailLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(len(vocabulary), 128, padding_idx=0)
        self.lstm = nn.LSTM(128, 128, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.output = nn.Linear(128, 1)

    def forward(self, tokens, lengths):
        sequence, _ = self.lstm(self.embedding(tokens))
        rows = torch.arange(len(lengths), device=tokens.device)
        return self.output(self.dropout(sequence[rows, lengths - 1])).squeeze(1)

def predict(model, loader, device):
    model.eval()
    probabilities, labels = [], []
    with torch.no_grad():
        for tokens, lengths, batch_labels in loader:
            logits = model(tokens.to(device), lengths.to(device))
            probabilities.extend(torch.sigmoid(logits).cpu().numpy())
            labels.extend(batch_labels.numpy())
    return np.asarray(labels, dtype=int), np.asarray(probabilities)

In [ ]:
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(EmailDataset(train), batch_size=BATCH_SIZE, shuffle=True, generator=generator)
validation_loader = DataLoader(EmailDataset(validation), batch_size=BATCH_SIZE)
test_loader = DataLoader(EmailDataset(test), batch_size=BATCH_SIZE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lstm_model = EmailLSTM().to(device)
positive_weight = torch.tensor([(len(train) - train["label"].sum()) / train["label"].sum()], device=device)
loss_function = nn.BCEWithLogitsLoss(pos_weight=positive_weight)
optimizer = torch.optim.AdamW(lstm_model.parameters(), lr=0.001)

history = []
best_validation_f1 = -1
for epoch in range(1, LSTM_EPOCHS + 1):
    lstm_model.train()
    total_loss = 0
    for tokens, lengths, labels in train_loader:
        tokens, lengths, labels = tokens.to(device), lengths.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = loss_function(lstm_model(tokens, lengths), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)

    validation_labels, validation_probabilities = predict(lstm_model, validation_loader, device)
    validation_f1 = f1_score(validation_labels, validation_probabilities >= 0.5)
    history.append({"epoch": epoch, "training_loss": total_loss / len(train), "validation_f1": validation_f1})
    if validation_f1 > best_validation_f1:
        best_validation_f1 = validation_f1
        best_state = {name: value.detach().cpu().clone() for name, value in lstm_model.state_dict().items()}

lstm_model.load_state_dict(best_state)
history_frame = pd.DataFrame(history)
display(history_frame)
history_frame.set_index("epoch").plot(subplots=True, figsize=(7, 5), title=["Training loss", "Validation F1"])
plt.tight_layout()
plt.savefig(RESULTS_DIR / "figures/lstm-training-history.png", dpi=180)
plt.show()

test_labels, lstm_probabilities = predict(lstm_model, test_loader, device)
lstm_metrics = evaluate("LSTM", test_labels, lstm_probabilities, "lstm")
lstm_metrics.update({"best_validation_f1": best_validation_f1, "epochs": LSTM_EPOCHS, "device": str(device)})
(RESULTS_DIR / "metrics/lstm.json").write_text(json.dumps(lstm_metrics, indent=2))
torch.save({"model_state": best_state, "vocabulary": vocabulary}, MODELS_DIR / "lstm.pt")
display(pd.Series(lstm_metrics).loc[["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]].to_frame("score"))

## 7. Compare and export results

In [ ]:
metric_names = ["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]
comparison = pd.DataFrame([
    {"model": "TF-IDF Logistic Regression", **{metric: classic_metrics[metric] for metric in metric_names}},
    {"model": "LSTM", **{metric: lstm_metrics[metric] for metric in metric_names}},
])
comparison.to_csv(RESULTS_DIR / "model-comparison.csv", index=False)

summary = {
    "records_after_deduplication": len(emails),
    "train_records": len(train),
    "validation_records": len(validation),
    "test_records": len(test),
    "ham": int((emails["label"] == 0).sum()),
    "spam": int((emails["label"] == 1).sum()),
    "lstm_epochs": LSTM_EPOCHS,
}
(RESULTS_DIR / "run-summary.json").write_text(json.dumps(summary, indent=2))
display(comparison.style.format({metric: "{:.4f}" for metric in metric_names}).highlight_max(subset=metric_names, color="#d9ead3"))

if COLAB:
    export_dir = ROOT / "section_01_export"
    shutil.copytree(RESULTS_DIR, export_dir / "reports", dirs_exist_ok=True)
    shutil.copytree(MODELS_DIR, export_dir / "models", dirs_exist_ok=True)
    shutil.make_archive("/content/section_01_results", "zip", root_dir=export_dir)

## Interpretation

- Compare precision, recall, F1, ROC AUC, and average precision rather than accuracy alone.
- A false positive blocks a legitimate message; a false negative allows an unwanted message through.
- Compare both models on the same held-out test records.
- SpamAssassin contains spam/ham labels, not explicit phishing labels, and is an old corpus. Results therefore demonstrate the requested workflow but do not establish modern phishing-detection performance.